In [ ]:

import os, sys, glob, shutil, time, json
import numpy as np, pandas as pd
from multiprocessing import Pool

t_start = time.perf_counter()
def log(m): print(f"[{time.perf_counter()-t_start:7.0f}с] {m}", flush=True)

base = os.path.dirname(glob.glob("/kaggle/input/**/attr_features.py", recursive=True)[0])
os.makedirs("/kaggle/working/src", exist_ok=True)
for p in glob.glob(base + "/*.py"): shutil.copy(p, "/kaggle/working/src/")
open("/kaggle/working/src/__init__.py", "a").close()
os.makedirs("/kaggle/working/models", exist_ok=True)
shutil.copy(base + "/anti_words.json", "/kaggle/working/models/anti_words.json")
os.chdir("/kaggle/working"); sys.path.insert(0, "/kaggle/working")

from src.attr_features import parse, compare, FEATURE_NAMES
from src.name_features import parse_name, compare_names, build_idf, NAME_FEATURE_NAMES
from src.string_features import compare_strings, STRING_FEATURE_NAMES
from src.neighbour_features import build as nb_build, compare as nb_compare, NEIGHBOUR_FEATURE_NAMES
from src.brand_features import colours, canonical, mine_aliases, compare_brands, compare_colours, BRAND_FEATURE_NAMES
from src.hybrid import product_disjoint_pair_masks
from src.metrics import macro_pr_auc
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import average_precision_score

items = pd.read_parquet(base + "/items_human.parquet")
matches = pd.read_parquet(base + "/matches.parquet", columns=["id1","id2","target"])
evalp = pd.read_parquet(base + "/eval_pairs.parquet")
log(f"товаров {len(items):,}, пар {len(matches):,}, линейка {len(evalp):,}")

ID = items["id"].to_numpy(); NAME = items["name"].astype(str).tolist()
ATTR = items["attributes"].tolist(); CAT = items["category"].astype(str).to_numpy()

cards = {int(i): parse(n, a, name=n) for i, n, a in zip(ID, NAME, ATTR)}
log("attr: карточки разобраны")
names = {int(i): parse_name(n) for i, n in zip(ID, NAME)}
idf, avg = build_idf(list(names.values()))
log(f"name: разобрано, слов в IDF {len(idf):,}")
cols = {int(i): colours(n + " " + str(a)) for i, n, a in zip(ID, NAME, ATTR)}
brand = {int(i): frozenset(x for x in (canonical(v) for v in c.slots.get("brand", ())) if x)
         for i, c in cards.items()}
log("цвета и бренды разобраны")

tm, vm = product_disjoint_pair_masks(matches["id1"].to_numpy(), matches["id2"].to_numpy(), 0, 3)
brand_one = {i: (sorted(b)[0] if b else "") for i, b in brand.items()}
aliases = mine_aliases(
    ((int(a), int(b), int(t)) for a, b, t in
     zip(matches["id1"].to_numpy()[tm], matches["id2"].to_numpy()[tm], matches["target"].to_numpy()[tm])),
    brand_one)
json.dump(aliases, open("/kaggle/working/brand_aliases.json", "w"), ensure_ascii=False)
log(f"алиасов брендов добыто: {len(aliases):,}")

profile = nb_build(items[["id","name","category"]])
log("окрестности построены")

# Косинус пары считается по той же матрице TF-IDF, что и окрестности — иначе величина и
# её отсчёт от окрестности окажутся в разных шкалах.
sim_of = {}
for cat, group in items.groupby("category", sort=False):
    gid = group["id"].to_numpy()
    M = TfidfVectorizer(min_df=1, sublinear_tf=True).fit_transform(group["name"].astype(str).tolist())
    pos = {int(x): r for r, x in enumerate(gid)}
    sim_of[cat] = (M, pos)
log("матрицы TF-IDF по категориям готовы")

cat_of = dict(zip(ID.tolist(), CAT.tolist()))
def pair_sim(a, b):
    ca = cat_of.get(a); 
    if ca is None or ca != cat_of.get(b): return 0.0
    M, pos = sim_of[ca]
    if a not in pos or b not in pos: return 0.0
    return float((M[pos[a]] @ M[pos[b]].T).toarray()[0, 0])

ALL = FEATURE_NAMES + NAME_FEATURE_NAMES + STRING_FEATURE_NAMES + NEIGHBOUR_FEATURE_NAMES + BRAND_FEATURE_NAMES
def row(pair):
    a, b = pair
    d = compare(cards[a], cards[b])
    d.update(compare_names(names[a], names[b], idf, avg))
    d.update(compare_strings(NAME[POS[a]], NAME[POS[b]]))
    d.update(nb_compare(a, b, pair_sim(a, b), profile))
    d.update(compare_brands(brand[a], brand[b], aliases))
    d.update(compare_colours(cols[a], cols[b]))
    return [d[k] for k in ALL]
POS = {int(x): r for r, x in enumerate(ID)}

def build_matrix(frame, tag):
    pairs = list(zip(frame["id1"].astype(int), frame["id2"].astype(int)))
    t0 = time.perf_counter()
    out = np.array([row(p) for p in pairs], dtype=np.float32)
    log(f"{tag}: {len(pairs):,} пар за {time.perf_counter()-t0:.0f}с "
        f"({len(pairs)/(time.perf_counter()-t0):.0f} пар/с)")
    return out

X = build_matrix(matches, "обучающие")
np.save("/kaggle/working/features_new.npy", X)
E = build_matrix(evalp, "линейка")
np.save("/kaggle/working/features_eval.npy", E)
json.dump(list(ALL), open("/kaggle/working/feature_names.json","w"), ensure_ascii=False)

y = matches["target"].to_numpy(np.int8)
cat_pair = matches["id1"].map(cat_of).astype(str).to_numpy()
tr, va = np.flatnonzero(tm), np.flatnonzero(vm)
S = np.load(base + "/human_features_v13.npy")
ye = evalp["target"].to_numpy(np.int8); ce = evalp["category"].astype(str).to_numpy()
def macro_eval(p):
    return float(np.mean([average_precision_score(ye[ce==k], p[ce==k]) for k in np.unique(ce)
                          if len(np.unique(ye[ce==k]))>1]))

log("обучение")
Se = None
for tag, Xtr, Xev in (("168 старых", S, None), ("97 новых", X, E),
                      ("168 + 97", np.hstack([S, X]), None)):
    clf = HistGradientBoostingClassifier(max_iter=400, learning_rate=0.08, random_state=0)
    clf.fit(Xtr[tr], y[tr])
    p = clf.predict_proba(Xtr[va])[:, 1]
    line = f"{tag:<12} holdout {macro_pr_auc(y[va], p, cat_pair[va])[0]:.6f}"
    if Xev is not None:
        pe = clf.predict_proba(Xev)[:, 1]
        np.save("/kaggle/working/eval_scores_new.npy", pe)
        line += f"   линейка {macro_eval(pe):.6f}"
    log(line)
log("готово")
